# 04 — Startup Panel Construction

**Author:** Ricardo Sanchez  
**Project:** Monetary Policy Shocks and Startup Funding Transitions (Seed → Series A)  
**Data Source:** Crunchbase (cleaned) | FRED | Monetary Policy Shock Series

---

## Objective

Combine the cleaned FinTech and AI survival datasets into a single startup × quarter **person-period panel** suitable for discrete-time hazard modeling. Each row represents one firm observed in one calendar quarter. The panel is then enriched with time-varying macro covariates (monetary policy shocks, Core CPI, Real GDP), lagged shock terms, and pre-computed feature transformations shared across all downstream models.

## Pipeline Overview

| Stage | Description |
|-------|-------------|
| **Part 1** | Helper functions for quarter arithmetic |
| **Part 2** | Load and union FinTech + AI survival datasets |
| **Part 3** | Expand to startup × quarter person-period format |
| **Part 4** | Merge quarterly macro variables (shocks, CPI, GDP) |
| **Part 5** | Create shock lags (L1–L5) |
| **Part 6** | Feature engineering — shared covariate transforms |
| **Part 7** | Export final panel |

## Inputs
- `data/cleaned/cleaned_fintech/fintech_simple_survival.csv`
- `data/cleaned/cleaned_ai/ai_simple_survival.csv`
- `data/raw/macro/monetary_shock.csv`
- `data/raw/macro/core_cpi.csv`
- `data/raw/macro/real_gdp.csv`

## Outputs
- `data/cleaned/discrete_time_hazard_data/startup_seriesA_person_period_quarterly_final_all.csv`

In [1]:
import pandas as pd
import numpy as np

In [ ]:
# ── File paths ──
FINTECH_PATH = r"..\data\cleaned\cleaned_fintech\fintech_simple_survival.csv"
AI_PATH = r"..\data\cleaned\cleaned_ai\ai_simple_survival.csv"
SHOCK_PATH = r"..\data\raw\macro\monetary_shock.csv"
CORE_CPI_PATH = r"..\data\raw\macro\core_cpi.csv"
REAL_GDP_PATH = r"..\data\raw\macro\real_gdp.csv"
OUT_PATH = r"..\data\cleaned\discrete_time_hazard_data\startup_seriesA_person_period_quarterly_final_all.csv"

In [ ]:
# ── Panel settings ──
ID_COL = "firm_id"
SEED_COL = "seed_date"
SERIESA_COL = "series_a_date"

OBS_START = pd.Timestamp("2016-01-01")
OBS_END   = pd.Timestamp("2023-09-30")  # dataset censor
KEEP_MAX_PERIOD_START = pd.Timestamp("2023-07-01")  # keep 2023-Q3 row

---

## Part 1 — Helper Functions

Utility functions for converting between quarter labels and timestamps, and for quarter-level date arithmetic.

In [4]:
# Convert a quarter label (e.g., '2018-Q4') into a Timestamp for the start of that quarter
def qlabel_to_qstart(s):
    s = str(s).strip().replace("-", "")
    return pd.Period(s, freq="Q-DEC").to_timestamp(how="S")

In [5]:
# Return the first day of the calendar quarter containing ts (Q1=Jan–Mar, Q2=Apr–Jun, Q3=Jul–Sep, Q4=Oct–Dec)
def qstart(ts):
    q = (ts.month - 1)//3
    return pd.Timestamp(ts.year, 1 + 3*q, 1)

In [6]:
# Return the last day of the calendar quarter containing ts (handles month lengths & leap years)
def qend(ts):
    q = (ts.month - 1)//3
    m = 3 + 3*q
    return pd.Timestamp(ts.year, 12, 31) if m==12 else pd.Timestamp(ts.year, m+1, 1) - pd.Timedelta(days=1)

In [7]:
# Shift timestamp ts by k calendar quarters (Jan/Apr/Jul/Oct starts); supports negative k
def add_quarters(ts, k):
    return ts + pd.offsets.QuarterBegin(startingMonth=1)*k

In [ ]:
# Defensive column renaming — handles minor naming variants between FinTech and AI files
def light_remap(df):
    cols = {c.lower(): c for c in df.columns}
    # map common variants if your AI file uses slightly different names
    for srcs, dst in [
        (["company_id","startup_id","id"], "firm_id"),
        (["seed","seed_round_date","date_seed"], "seed_date"),
        (["series_a","seriesa_date","series_a_round_date","date_series_a"], "series_a_date"),
        (["exit","exitdate"], "exit_date"),
    ]:
        for s in srcs:
            if s in cols and dst not in df.columns:
                df.rename(columns={cols[s]: dst}, inplace=True)
                break
    return df

---

## Part 2 — Load and Union Startups

Load the cleaned FinTech and AI survival CSVs (produced by notebooks 02 and 03), align their columns, and concatenate into a single cross-sector dataset.

In [ ]:
# Load cleaned sector survival datasets
fin = pd.read_csv(FINTECH_PATH)
ai = pd.read_csv(AI_PATH)

In [ ]:
# Align columns across both datasets (handles any column mismatches)
all_cols = list(set(fin.columns).union(ai.columns))

# Reindex so both dataframes share the same column set (missing cols filled with NaN)
fin = fin.reindex(columns=all_cols)
ai = ai.reindex(columns=all_cols)

# Concatenate into a single cross-sector dataset
both = pd.concat([fin,ai], ignore_index=True)

In [ ]:
# Ensure the firm identifier column exists; create a temporary row ID if missing
if ID_COL not in both.columns:
    both["_row_id"] = np.arange(len(both)); ID_COL = "_row_id" # temporary ID column

In [ ]:
# Parse date columns to datetime for time-based operations
both[SEED_COL] = pd.to_datetime(both[SEED_COL], errors="coerce")
both[SERIESA_COL] = pd.to_datetime(both[SERIESA_COL], errors="coerce")


In [ ]:
# Drop firms with missing seed dates (cannot enter the risk set)
both = both[both[SEED_COL].notna()].copy()

In [ ]:
# ── Define entry, event, and censoring dates ──

# Entry: the later of seed date or observation window start
both["_entry_date"] = both[SEED_COL].apply(lambda d: max(d,OBS_START))

# Event: Series A date (NaT if no Series A observed)
both["_event_date"] = both[SERIESA_COL]

# Follow-up end: event date if observed, otherwise censored at observation window end
both["_followup_end"] = both["_event_date"]
both["_followup_end"] = both["_followup_end"].fillna(OBS_END).apply(lambda d: min(d, OBS_END))

# Keep only firms whose entry date falls before their follow-up end
both = both[both["_entry_date"] <= both["_followup_end"]].copy()


---

## Part 3 — Expand to Startup × Quarter Panel

Create one row per firm per calendar quarter, from entry through event or censoring. The `event_seriesA` indicator is 1 only in the quarter the Series A occurs; all prior quarters are 0. Expansion stops immediately after the event quarter.

In [ ]:
# ── Person-period expansion loop ──
rows = []


for _, r in both.iterrows():
    sid = r[ID_COL] # firm ID
    q = qstart(r["_entry_date"])  # start at entry quarter
    end_q = qend(r["_followup_end"])  # end at followup end quarter
    event_q = qstart(r["_event_date"]) if pd.notna(r["_event_date"]) else pd.NaT # event quarter
    k=0 # quarter counter

    while q <= end_q:
        event = int(pd.notna(event_q) and q == event_q) # event occurs this quarter
        rows.append({
            ID_COL:sid,
            "period_start": q, 
            "year": q.year,
            "quarter": ((q.month - 1) //3) +1, # Compute the calendar quarter number (1–4) from the month value of q
            "duration_q": k,
            "event_seriesA": event
        })
        if event==1: break  # stop after event
        q = add_quarters(q,1); k+=1 # next quarter

In [ ]:
# Assemble the panel dataframe and merge back firm-level covariates
panel = pd.DataFrame(rows)

# Left-join startup covariates (drop internal working columns)
panel = panel.merge(
    both.drop(columns=["_entry_date","_event_date","_followup_end"]),
    on=ID_COL, how="left"
)
panel.head()

In [ ]:
# Trim panel to the observation window (2016 Q1 through 2023 Q3)
panel = panel[(panel["period_start"] >= OBS_START) & (panel["period_start"] <= KEEP_MAX_PERIOD_START)].copy()

---

## Part 4 — Merge Quarterly Macro Variables

Attach time-varying macroeconomic covariates to each panel quarter. All three series are merged on `period_start` (first day of the calendar quarter).

In [ ]:
# Load monetary policy shock series and align quarter labels to timestamps
shock = pd.read_csv(SHOCK_PATH)
shock["period_start"] = shock["quarter_year"].map(qlabel_to_qstart)

shock.head()


,quarter_year,mps_shock,period_start
0,2014-Q1,0.0339,2014-01-01
1,2014-Q2,0.0283,2014-04-01
2,2014-Q3,0.0273,2014-07-01
3,2014-Q4,-0.0199,2014-10-01
4,2015-Q1,-0.0662,2015-01-01


In [ ]:
# Convert raw shock (percentage points) to 25-basis-point units for interpretability
shock["shock_25bp_units"] = shock["mps_shock"] / 0.25 
shock.head(10)

,quarter_year,mps_shock,period_start,shock_25bp_units
0,2014-Q1,0.0339,2014-01-01,0.1356
1,2014-Q2,0.0283,2014-04-01,0.1132
2,2014-Q3,0.0273,2014-07-01,0.1092
3,2014-Q4,-0.0199,2014-10-01,-0.0796
4,2015-Q1,-0.0662,2015-01-01,-0.2648
5,2015-Q2,-0.0324,2015-04-01,-0.1296
6,2015-Q3,-0.0616,2015-07-01,-0.2464
7,2015-Q4,0.0819,2015-10-01,0.3276
8,2016-Q1,-0.1054,2016-01-01,-0.4216
9,2016-Q2,-0.0215,2016-04-01,-0.0860


In [ ]:
# Merge monetary shocks into the panel on quarter
panel = panel.merge(shock[["period_start", "shock_25bp_units"]],
                    on="period_start", 
                    how="left")

### 4.2 Core CPI

In [ ]:
# Load Core CPI (year-over-year % change, excluding food & energy)
core_cpi = pd.read_csv(CORE_CPI_PATH)
core_cpi["period_start"] = core_cpi["observation_date"].map(qlabel_to_qstart)

In [ ]:
# Rename to analysis-friendly column name
core_cpi["core_cpi_yoy"] = pd.to_numeric(core_cpi["CPILFESL_PC1"], errors="coerce")

In [ ]:
# Merge Core CPI into the panel on quarter
panel = panel.merge(core_cpi[["period_start","core_cpi_yoy"]], on="period_start", how="left")

### 4.3 Real GDP

In [ ]:
# Load Real GDP growth (year-over-year % change)
real_gdp = pd.read_csv(REAL_GDP_PATH)
real_gdp["period_start"] = real_gdp["observation_date"].map(qlabel_to_qstart)

# Rename to analysis-friendly column name
real_gdp["real_gdp_yoy"] = pd.to_numeric(real_gdp["GDPC1_PC1"], errors="coerce")

In [ ]:
# Merge Real GDP into the panel on quarter
panel = panel.merge(real_gdp[["period_start","real_gdp_yoy"]], on="period_start", how="left")

---

## Part 5 — Shock Lags

Create lagged monetary policy shock terms (L1 through L5) within each firm's time series. These capture the delayed transmission of monetary policy to venture capital deployment.

In [ ]:
# Sort by firm and quarter, then compute within-firm lags of the shock variable
panel = panel.sort_values([ID_COL, "period_start"])

for L in range(1,6):
    panel[f"shock25_lag{L}"] = panel.groupby(ID_COL)["shock_25bp_units"].shift(L)

---

## Part 6 — Feature Engineering

Pre-compute covariate transformations that are shared across all downstream models. Performing these here (rather than in the model notebook) ensures consistency across specifications and avoids duplicating transformation logic.

### 6.1 Seed Funding Amount

In [ ]:
# Coerce seed_amount_usd to numeric (handles dollar-formatted strings if present)
panel["seed_amount_usd"] = pd.to_numeric(
    panel["seed_amount_usd"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.strip(),
    errors="coerce"
)

# Winsorize at 1st and 99th percentiles to limit outlier influence
low, high = panel["seed_amount_usd"].quantile([0.01, 0.99])
panel["seed_amount_usd_w"] = panel["seed_amount_usd"].clip(lower=low, upper=high)

# Log-transform (log1p handles zero values gracefully)
panel["log_seed_amt"] = np.log1p(panel["seed_amount_usd_w"])

### 6.2 Partner Investor Count

In [ ]:
# Coerce to numeric (handles any non-numeric entries from raw Crunchbase data)
panel["num_partner_investors"] = pd.to_numeric(panel["num_partner_investors"], errors="coerce")

# Log-transform with zero-fill for missing values (log1p(0) = 0)
panel["log_npi"] = np.log1p(panel["num_partner_investors"].fillna(0))

### 6.3 Duration Bins

Bin the continuous `duration_q` (quarters since entry) into 5 categories for the piecewise-constant baseline hazard. The bin boundaries reflect natural breakpoints in startup funding timelines.

In [ ]:
# Create duration bins: 0-2, 3-5, 6-8, 9-12, 13+ quarters
bins = [-1, 2, 5, 8, 12, 10**9]
labels = ["dur_b0_2", "dur_b3_5", "dur_b6_8", "dur_b9_12", "dur_b13p"]
panel["dur_bin"] = pd.cut(panel["duration_q"], bins=bins, labels=labels)

---

## Part 7 — Export

Save the final person-period panel with all macro merges, shock lags, and feature transformations applied.

In [ ]:
# Sort and export the final panel
panel.sort_values([ID_COL,"period_start"], inplace=True)
panel.to_csv(OUT_PATH, index=False)
print("Final combined panel saved:", OUT_PATH)

Final combined panel saved: C:\Users\Ricardo\Documents\Projects\capstone_project\data\cleaned\discrete_time_hazard_data\startup_seriesA_person_period_quarterly_final_all.csv


In [ ]:
# Verify final column set
print(panel.columns)

Index(['firm_id', 'period_start', 'year', 'quarter', 'duration_q',
       'event_seriesA', 'seed_year', 'time_to_series_a_months',
       'num_investors_total', 'seed_date', 'series_a_date', 'end_date',
       'event_series_a', 'seed_amount_usd', 'diversity_spotlight_flag',
       'num_partner_investors', 'metro_region', 'sector', 'entry_date',
       'shock_25bp_units', 'nasdaqcom_ccr', 't10y2y', 'shock25_lag1',
       'shock25_lag2', 'shock25_lag3', 'shock25_lag4', 'shock25_lag5'],
      dtype='object')


In [ ]:
# Display the first few rows of the final panel to verify the structure and merged data
print(panel.head())